# Notebook 01: From Bits to Qubits

**Quantum Cognition** — a curriculum bridging quantum computing and computational cognitive neuroscience.

---

## What you'll learn

1. How **coin flips** generalise from probability vectors to **superpositions**
2. What a **density matrix** is — and why it's the right generalisation of "beliefs"
3. **Von Neumann entropy** as the quantum version of Shannon entropy
4. Why **off-diagonal elements** (coherence) are the signature of genuinely quantum behaviour

### Prerequisites
- Linear algebra (matrix multiplication, eigenvalues)
- Probability (Bayes' rule, entropy) — e.g. spinning-up-alf NB 08
- **No quantum mechanics required** — we build everything from scratch

### Notation
This notebook uses **matrix notation only**. We'll introduce Dirac bra-ket notation in NB 03.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

# Our library
from quantum_cognition.core import (
    computational_basis, plus_state, minus_state, bell_state,
    pure_state_density_matrix, maximally_mixed,
    von_neumann_entropy, purity, fidelity, partial_trace,
    quantum_mutual_information,
)
from quantum_cognition.viz.bloch import plot_bloch_sphere, bloch_coordinates

plt.rcParams['figure.dpi'] = 120
print(f"JAX version: {jax.__version__}")
print(f"Backend: {jax.default_backend()}")

## 1. Classical Bits and Probability Vectors

A **fair coin** has two outcomes. We represent our uncertainty as a probability vector:

$$\mathbf{p} = \begin{pmatrix} p(\text{heads}) \\ p(\text{tails}) \end{pmatrix} = \begin{pmatrix} 0.5 \\ 0.5 \end{pmatrix}$$

Constraints: entries are real, non-negative, and sum to 1.

If we **know** the coin landed heads, the vector collapses to a **definite state**:

$$\mathbf{e}_0 = \begin{pmatrix} 1 \\ 0 \end{pmatrix}$$

This is just a standard basis vector. Nothing quantum yet.

In [ ]:
# Classical probability vectors
p_fair = jnp.array([0.5, 0.5])   # fair coin
p_heads = jnp.array([1.0, 0.0])  # definitely heads

print("Fair coin:", p_fair, "  sum =", p_fair.sum())
print("Definite heads:", p_heads, "  sum =", p_heads.sum())

# Shannon entropy: H(p) = -sum(p * log(p))
def shannon_entropy(p):
    """Shannon entropy, handling zeros gracefully."""
    p_safe = jnp.clip(p, 1e-12, None)
    return -jnp.sum(p * jnp.log(p_safe))

print(f"\nShannon entropy (fair coin): {shannon_entropy(p_fair):.4f}  (= ln 2 = {jnp.log(2.0):.4f})")
print(f"Shannon entropy (definite):  {shannon_entropy(p_heads):.4f}  (no uncertainty)")

## 2. The Quantum Leap: Amplitudes and Superposition

Now for the key generalisation. Instead of probabilities, we allow **complex-valued amplitudes**:

$$|\psi\rangle = \begin{pmatrix} \alpha \\ \beta \end{pmatrix}, \quad \alpha, \beta \in \mathbb{C}, \quad |\alpha|^2 + |\beta|^2 = 1$$

The probability of measuring outcome 0 is $|\alpha|^2$, and outcome 1 is $|\beta|^2$.

**What's new?** The amplitudes can be complex and can have **relative phase**. Two states that give identical measurement probabilities can still be physically different:

$$|+\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ 1 \end{pmatrix} \qquad |-\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ -1 \end{pmatrix}$$

Both give 50/50 when measured — just like a fair coin. But the minus sign in $|-\rangle$ leads to **interference** when these states combine. This is the entire story of quantum cognition in one sentence: *interference between cognitive paths can make quantum models behave differently from classical ones.*

In [ ]:
# Quantum state vectors
psi_0 = computational_basis(2, 0)  # |0⟩ — "definitely heads"
psi_1 = computational_basis(2, 1)  # |1⟩ — "definitely tails"
psi_plus = plus_state()            # |+⟩ — equal superposition
psi_minus = minus_state()          # |−⟩ — equal superposition with phase flip

print("|0⟩ =", psi_0)
print("|+⟩ =", psi_plus)
print("|−⟩ =", psi_minus)

# Both |+⟩ and |−⟩ give the same measurement probabilities:
print(f"\nP(0) for |+⟩: {abs(psi_plus[0])**2:.2f},  P(1): {abs(psi_plus[1])**2:.2f}")
print(f"P(0) for |−⟩: {abs(psi_minus[0])**2:.2f},  P(1): {abs(psi_minus[1])**2:.2f}")
print("Same probabilities — but the states are different!")

### The Bloch Sphere

A single qubit's state can be visualised as a point on a sphere. The north pole is $|0\rangle$, the south pole is $|1\rangle$, and the equator contains all equal-superposition states like $|+\rangle$ and $|-\rangle$.

Pure states live on the surface. Mixed states (uncertain mixtures) live inside.

In [ ]:
# Visualise key states on the Bloch sphere
states = [
    pure_state_density_matrix(psi_0),
    pure_state_density_matrix(psi_1),
    pure_state_density_matrix(psi_plus),
    pure_state_density_matrix(psi_minus),
    maximally_mixed(2),  # centre of sphere
]
labels = ["|0⟩", "|1⟩", "|+⟩", "|−⟩", "I/2 (mixed)"]

fig = plot_bloch_sphere(states, labels, title="Key Qubit States on the Bloch Sphere")
plt.tight_layout()
plt.show()

## 3. Density Matrices: Generalised Beliefs

A state vector $|\psi\rangle$ describes a **pure state** — maximal knowledge. But in cognitive science, we often deal with **mixtures**: "the agent is in state $|0\rangle$ with probability $p$ or state $|1\rangle$ with probability $1-p$."

The **density matrix** handles both cases:

$$\rho = |\psi\rangle\langle\psi| \quad \text{(pure state)}$$
$$\rho = \sum_i p_i |\psi_i\rangle\langle\psi_i| \quad \text{(mixed state)}$$

Properties of any valid density matrix:
- **Hermitian**: $\rho = \rho^\dagger$
- **Positive semi-definite**: all eigenvalues $\geq 0$
- **Trace one**: $\text{Tr}(\rho) = 1$
- **Purity**: $\text{Tr}(\rho^2) = 1$ for pure states, $< 1$ for mixed states

**The key insight for cognitive science:** A classical belief vector $\mathbf{p} = (p_1, \ldots, p_n)$ maps to a **diagonal** density matrix $\rho = \text{diag}(\mathbf{p})$. The off-diagonal elements are what make the quantum version strictly more expressive — they encode **coherences** (correlations between alternatives that have no classical analogue).

In [ ]:
# Pure state density matrices
rho_0 = pure_state_density_matrix(psi_0)
rho_plus = pure_state_density_matrix(psi_plus)

print("ρ for |0⟩ (pure, definite):")
print(rho_0)
print(f"  Purity: {purity(rho_0):.4f}")

print("\nρ for |+⟩ (pure, superposition):")
print(rho_plus)
print(f"  Purity: {purity(rho_plus):.4f}")

# Classical beliefs as a diagonal density matrix
from quantum_cognition.models.bridge import beliefs_to_density_matrix
beliefs = jnp.array([0.7, 0.3])
rho_classical = beliefs_to_density_matrix(beliefs)

print("\nρ for classical beliefs [0.7, 0.3] (diagonal — no coherence):")
print(rho_classical)
print(f"  Purity: {purity(rho_classical):.4f}  (< 1, it's mixed)")

# Maximally mixed state
rho_mixed = maximally_mixed(2)
print("\nρ for maximally mixed state I/2 (maximum ignorance):")
print(rho_mixed)
print(f"  Purity: {purity(rho_mixed):.4f}  (= 1/d = 0.5)")

### Notice the difference

| State | Diagonal | Off-diagonal | Purity | Type |
|-------|----------|-------------|--------|------|
| $\|0\rangle$ | (1, 0) | 0 | 1.0 | Pure, definite |
| $\|+\rangle$ | (0.5, 0.5) | **0.5** | 1.0 | Pure, superposition |
| beliefs [0.7, 0.3] | (0.7, 0.3) | 0 | 0.58 | Mixed, classical |
| $I/2$ | (0.5, 0.5) | 0 | 0.5 | Mixed, maximally uncertain |

The $|+\rangle$ state and the maximally mixed state $I/2$ have **identical diagonals** (both 50/50). But $|+\rangle$ is pure (purity = 1) while $I/2$ is maximally mixed (purity = 0.5). The difference is entirely in the **off-diagonal coherence terms**.

This is why density matrices are the right tool for quantum cognition: they can represent states that *look* classically identical but *behave* differently under evolution.

## 4. Von Neumann Entropy: Quantum Uncertainty

Shannon entropy measures uncertainty in a classical probability distribution:

$$H(\mathbf{p}) = -\sum_i p_i \ln p_i$$

Von Neumann entropy is the direct generalisation for density matrices:

$$S(\rho) = -\text{Tr}(\rho \ln \rho) = -\sum_i \lambda_i \ln \lambda_i$$

where $\lambda_i$ are the eigenvalues of $\rho$.

**The classical-quantum bridge:** When $\rho$ is diagonal (classical beliefs), von Neumann entropy **exactly equals** Shannon entropy. The quantum version is strictly more general.

In [ ]:
# Von Neumann entropy for various states
states_and_names = [
    (pure_state_density_matrix(psi_0), "|0⟩ (pure, definite)"),
    (pure_state_density_matrix(psi_plus), "|+⟩ (pure, superposition)"),
    (beliefs_to_density_matrix(jnp.array([0.7, 0.3])), "beliefs [0.7, 0.3]"),
    (maximally_mixed(2), "I/2 (maximally mixed)"),
]

print("State                        S(ρ)     Shannon H(diag)")
print("-" * 55)
for rho, name in states_and_names:
    S = von_neumann_entropy(rho)
    diag = jnp.diag(rho).real
    H = shannon_entropy(diag)
    print(f"{name:30s} {S:6.4f}   {H:6.4f}")

print(f"\nln(2) = {jnp.log(2.0):.4f}  (maximum entropy for d=2)")

**Key observation:** The $|+\rangle$ state has $S(\rho) = 0$ (pure state, zero von Neumann entropy) but Shannon entropy of its diagonal is $\ln 2$ (maximum uncertainty). This is because $|+\rangle$ is a *definite* quantum state — we know *exactly* which superposition we're in. The uncertainty only appears when we *measure* in the computational basis.

This is the fundamental distinction:
- **Shannon entropy of diagonal** = uncertainty about *measurement outcomes*
- **Von Neumann entropy** = uncertainty about the *state itself*

For classical beliefs (diagonal $\rho$), these are the same. For quantum states with coherence, they diverge.

## 5. Entanglement and Correlations

When two quantum systems interact, they can become **entangled** — correlated in a way that has no classical analogue.

A **Bell state** is the simplest example:

$$|\Phi^+\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ 0 \\ 0 \\ 1 \end{pmatrix}$$

This is a 2-qubit state (lives in a 4-dimensional space). If you measure the first qubit and get 0, the second *must* also be 0. If you get 1, the second must be 1. The outcomes are perfectly correlated.

**The cognitive analogue:** Entanglement formalises situations where beliefs about one variable are *inseparable* from beliefs about another — not just correlated, but fundamentally intertwined. This is relevant for modelling context effects in decision-making.

In [ ]:
# Bell state: maximally entangled 2-qubit state
psi_bell = bell_state("phi+")
rho_bell = pure_state_density_matrix(psi_bell)

print("Bell state |Φ+⟩:", psi_bell)
print("\nFull density matrix (4×4):")
print(jnp.round(rho_bell, 3))
print(f"\nVon Neumann entropy of full state: {von_neumann_entropy(rho_bell):.4f}  (pure!)")

# Trace out second qubit to get reduced state of first qubit
rho_A = partial_trace(rho_bell, (2, 2), trace_out=1)
print(f"\nReduced state of qubit A (after tracing out B):")
print(jnp.round(rho_A, 3))
print(f"Von Neumann entropy of qubit A: {von_neumann_entropy(rho_A):.4f}  (= ln 2, maximally mixed!)")

# Quantum mutual information: how much are A and B correlated?
mi = quantum_mutual_information(rho_bell, (2, 2))
print(f"\nQuantum mutual information I(A:B): {mi:.4f}  (= 2 ln 2 = {2*jnp.log(2.0):.4f})")
print("This is the maximum possible for two qubits — they are maximally entangled.")

### The entanglement paradox

The full Bell state is **pure** ($S = 0$) — we have *complete* knowledge of the joint system. But each qubit *individually* is **maximally mixed** ($S = \ln 2$) — complete ignorance. All the information is in the *correlations*, not the individual parts.

This is strikingly different from classical probability: classically, if you know everything about a joint distribution, you certainly know everything about the marginals. In quantum theory, the whole can be known while the parts are uncertain.

## 6. Interference: Why Quantum Models Are Different

The defining feature of quantum probability is **interference**. In classical probability, if there are two paths to an outcome, the probabilities add:

$$P(A) = P(A \text{ via path 1}) + P(A \text{ via path 2})$$

In quantum probability, the *amplitudes* add, and then we square:

$$P(A) = |\alpha_1 + \alpha_2|^2 = |\alpha_1|^2 + |\alpha_2|^2 + \underbrace{2\text{Re}(\alpha_1^* \alpha_2)}_{\text{interference term}}$$

The interference term can be **positive** (constructive) or **negative** (destructive). This is why quantum models can produce probability patterns that no classical model can — including violations of the [sure-thing principle](https://en.wikipedia.org/wiki/Sure-thing_principle) observed in human decision-making experiments.

In [ ]:
# Demonstrating interference
# Two paths with amplitudes that can constructively or destructively interfere

alpha_1 = 0.6 + 0.0j   # amplitude via path 1
alpha_2 = 0.4 + 0.0j   # amplitude via path 2 (same phase)

# Classical: probabilities add
P_classical = abs(alpha_1)**2 + abs(alpha_2)**2

# Quantum: amplitudes add, then square
P_constructive = abs(alpha_1 + alpha_2)**2
interference_constructive = 2 * (alpha_1.conjugate() * alpha_2).real

print("=== Constructive interference (same phase) ===")
print(f"Classical (no interference):  P = |α₁|² + |α₂|² = {P_classical:.4f}")
print(f"Quantum (amplitudes add):     P = |α₁ + α₂|² = {P_constructive:.4f}")
print(f"Interference term:            2Re(α₁*α₂) = {interference_constructive:+.4f}")

# Now with opposite phase
alpha_2_neg = -0.4 + 0.0j
P_destructive = abs(alpha_1 + alpha_2_neg)**2
interference_destructive = 2 * (alpha_1.conjugate() * alpha_2_neg).real

print(f"\n=== Destructive interference (opposite phase) ===")
print(f"Classical (no interference):  P = |α₁|² + |α₂|² = {P_classical:.4f}")
print(f"Quantum (amplitudes add):     P = |α₁ + α₂|² = {P_destructive:.4f}")
print(f"Interference term:            2Re(α₁*α₂) = {interference_destructive:+.4f}")

print("\n→ Same path probabilities, but interference changes the outcome!")

In [ ]:
# Visualise: interference shows up in the off-diagonal elements of ρ
# Build a density matrix with controlled coherence

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Vary coherence from 0 (classical) to 0.5 (maximal, pure |+⟩)
coherences = [0.0, 0.25, 0.5]
titles = ["No coherence\n(classical mix)", "Partial coherence\n(partially quantum)", "Full coherence\n(pure |+⟩)"]

for ax, c, title in zip(axes, coherences, titles):
    rho = jnp.array([[0.5, c], [c, 0.5]], dtype=jnp.complex64)
    S = von_neumann_entropy(rho)
    P = purity(rho)

    im = ax.imshow(np.abs(np.asarray(rho)), vmin=0, vmax=0.5, cmap='Blues')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['|0⟩', '|1⟩'])
    ax.set_yticklabels(['⟨0|', '⟨1|'])
    ax.set_title(f"{title}\nS={S:.3f}, Purity={P:.3f}", fontsize=10)

    # Annotate cells
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{abs(float(rho[i,j])):.2f}", ha='center', va='center', fontsize=14)

fig.colorbar(im, ax=axes, shrink=0.8, label='|ρᵢⱼ|')
fig.suptitle("Density Matrices: Classical to Quantum", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 7. Fidelity: How Close Are Two Quantum States?

When comparing models, we need a distance measure between states. The **fidelity** $F(\rho, \sigma)$ generalises the classical Bhattacharyya coefficient:

$$F(\rho, \sigma) = \left(\text{Tr}\sqrt{\sqrt{\rho}\,\sigma\,\sqrt{\rho}}\right)^2$$

- $F = 1$: states are identical
- $F = 0$: states are perfectly distinguishable (orthogonal)

For classical (diagonal) states, this reduces to $F(\mathbf{p}, \mathbf{q}) = \left(\sum_i \sqrt{p_i q_i}\right)^2$.

In [ ]:
# Fidelity between various states
rho_0 = pure_state_density_matrix(computational_basis(2, 0))
rho_1 = pure_state_density_matrix(computational_basis(2, 1))
rho_plus = pure_state_density_matrix(plus_state())
rho_mixed = maximally_mixed(2)

pairs = [
    (rho_0, rho_0, "|0⟩ vs |0⟩"),
    (rho_0, rho_1, "|0⟩ vs |1⟩"),
    (rho_0, rho_plus, "|0⟩ vs |+⟩"),
    (rho_plus, rho_mixed, "|+⟩ vs I/2"),
    (rho_0, rho_mixed, "|0⟩ vs I/2"),
]

print("State pair                F(ρ,σ)")
print("-" * 40)
for rho, sigma, name in pairs:
    F = fidelity(rho, sigma)
    print(f"{name:25s} {F:.4f}")

---

## Dual Perspective: Classical ↔ Quantum

| Classical Concept | Quantum Generalisation | Reduces to classical when... |
|---|---|---|
| Probability vector $\mathbf{p}$ | Density matrix $\rho$ | $\rho$ is diagonal |
| Shannon entropy $H(\mathbf{p})$ | Von Neumann entropy $S(\rho)$ | $\rho$ is diagonal |
| Bhattacharyya coefficient | Fidelity $F(\rho, \sigma)$ | Both $\rho, \sigma$ diagonal |
| Marginal distribution | Partial trace | No entanglement |
| Mutual information $I(X;Y)$ | Quantum MI $I(A:B)$ | No entanglement |
| Stochastic matrix $B$ | Unitary operator $U$ | See NB 03 (Szegedy) |

**The classical world is a special case of the quantum world.** Every classical model lives inside the quantum framework as a diagonal density matrix with no coherence. The quantum extension adds one new ingredient — off-diagonal elements — which enable interference.

---

## Exercises

### Basic

**Exercise 1.1:** Create a 3-level system (qutrit) in the state $|\psi\rangle = \frac{1}{\sqrt{3}}(|0\rangle + |1\rangle + |2\rangle)$. Compute its density matrix and verify that the purity is 1 and the von Neumann entropy is 0.

**Exercise 1.2:** Create a classical belief vector $\mathbf{p} = (0.5, 0.3, 0.2)$, convert it to a density matrix, and verify that the von Neumann entropy equals the Shannon entropy.

**Exercise 1.3:** For the Bell state $|\Psi^-\rangle = \frac{1}{\sqrt{2}}(|01\rangle - |10\rangle)$, compute the reduced density matrix of each qubit. Are they the same? Why?

### Stretch

**Exercise 1.4 (Interference requires coherence):** Prove computationally that for any **diagonal** 2×2 density matrix, applying a diagonal unitary $U = \text{diag}(e^{i\phi_1}, e^{i\phi_2})$ produces no change in measurement probabilities. Then show that for a density matrix **with** off-diagonal elements, the same unitary *does* change the probabilities. This is the mathematical core of why quantum cognition models can produce interference effects that classical models cannot.

**Exercise 1.5 (Decoherence):** Start with the pure state $|+\rangle$ and gradually zero out the off-diagonal elements of its density matrix (simulating decoherence). Plot how the von Neumann entropy changes as the state goes from pure to maximally mixed. Relate this to Khrennikov's conjecture that neural decoherence timescales determine when quantum-like effects are observable in cognition.

In [ ]:
# Scratch space for exercises
# Exercise 1.1
psi_qutrit = jnp.array([1, 1, 1], dtype=jnp.complex64) / jnp.sqrt(3.0)
rho_qutrit = pure_state_density_matrix(psi_qutrit)
print(f"Exercise 1.1: Purity = {purity(rho_qutrit):.4f}, S = {von_neumann_entropy(rho_qutrit):.4f}")

## Further Reading

- **Busemeyer & Bruza (2012)** — *Quantum Models of Cognition and Decision*. The foundational textbook.
- **Pothos & Busemeyer (2013)** — "Can quantum probability provide a new direction for cognitive modeling?" *Psychological Review*.
- **Nielsen & Chuang (2010)** — *Quantum Computation and Quantum Information*. Chapter 2 (density matrices, entropy) is the standard reference.
- **Khrennikov (2010)** — *Ubiquitous Quantum Structure*. The case for quantum-like models outside physics.

## Next: NB 02

In Notebook 02, we'll use these tools to build **quantum walks** — the quantum analogue of random walks — and connect them to **evidence accumulation** (drift-diffusion models) in cognitive neuroscience.